# BRACS TL Test: Tile-Level Evaluation

Tile-level evaluation of the breast carcinoma models on **BRACS** (external dataset), Full Rescaled & Filtered,
**TL test split**: 1,662,197 tiles (183,735 carcinoma positive). Only tile-level metrics are considered; slide-level
metrics are not interpretable here since the slides are represented by different (filtered) tiles.

Inputs are the per-tile prediction tables (`tables/carcinoma_prediction_table.json`) of the TL test runs
(experiment 170 on the external MLflow):

* **TL models** (tile classifiers: CNNs, GViT, frozen FMs), metrics logged as `test/<metric>`.
* **MIL models** (Classic / Hybrid MIL on PGP / Virchow2 embeddings), metrics logged as `tl_test/<metric>`.

All decisions use the **in-house-tuned tile thresholds** stored in each run's config; nothing is re-tuned on BRACS.

The notebook

1. compares the BRACS tile score distributions with the in-house (MMCI) test set, to explain the low BRACS recall
   despite high AUROC/AUPRC,
2. recomputes the tile-level metrics **without the DCIS tiles** (sanity-checked against the logged metrics on all
   tiles),
3. writes the main tables (logged metrics + run links) and the "Without DCIS tiles" sections of the result pages.

**Running:** kernel `Python (carcinoma)`. Set `EXTERNAL_MLFLOW_TRACKING_USERNAME` and
`EXTERNAL_MLFLOW_TRACKING_PASSWORD` (external MLflow credentials) before starting the kernel. The MMCI comparison
also needs the internal (cluster) MLflow, accessed with the kernel's default `MLFLOW_TRACKING_*` environment.

## Setup

In [ ]:
import os
import tempfile
from contextlib import contextmanager
from pathlib import Path

import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd
import yaml
from IPython.display import Markdown, display
from sklearn.metrics import (
    average_precision_score,
    precision_recall_curve,
    roc_auc_score,
)


# Internal (cluster) MLflow with the MMCI test runs; its credentials are the kernel's defaults,
# saved before they are overridden for the external server.
INTERNAL_MLFLOW_URI = "http://mlflow.rationai-mlflow:5000/"
MLFLOW_ENV_KEYS = ["MLFLOW_TRACKING_USERNAME", "MLFLOW_TRACKING_PASSWORD"]
INTERNAL_MLFLOW_ENV = {k: os.environ.get(k) for k in MLFLOW_ENV_KEYS}

# BRACS runs live on the external MLflow server; credentials come from the environment.
EXTERNAL_MLFLOW_URI = "https://mlflow.rationai.cloud.e-infra.cz"
os.environ["MLFLOW_TRACKING_USERNAME"] = os.environ["EXTERNAL_MLFLOW_TRACKING_USERNAME"]
os.environ["MLFLOW_TRACKING_PASSWORD"] = os.environ["EXTERNAL_MLFLOW_TRACKING_PASSWORD"]
mlflow.set_tracking_uri(EXTERNAL_MLFLOW_URI)

# Folder with the result pages (bracs_tl_eval.md, bracs_mil_eval.md) to update in place.
# None = only display the generated tables.
PAGES_DIR: Path | None = None

### Runs and data

Row names match the result pages (in page order). Each model has its BRACS TL test run (external MLflow), its model
registry version (used for the row links) and its in-house MMCI test run (internal MLflow).

In [ ]:
# page row name -> (BRACS TL test run id, model registry version, MMCI test run id)
TL_RUNS = {
    "VGG16": (
        "3976476c98254a0c87f92ee4417ec7fc",
        2,
        "fcc83051d3504ff0b035e8c2b3bc9d6e",
    ),
    "VGG16-stain": (
        "158a083e8a2744f0b2df1a58ededf1fb",
        7,
        "83c3a67c4c774f2bab560b1dcc37d2fe",
    ),
    "ResNet50": (
        "f6eb51e5ac264558a9e1b99ae68bc7c7",
        1,
        "0cde5922b5714a1ea5e43f648fd3e8da",
    ),
    "ResNet50-stain": (
        "6e905a507e4f44de9cf118ca2fb8c461",
        6,
        "475bec3c7c60457d9ebde5f2a6eb0bcf",
    ),
    "PGP (frozen)": (
        "9c2ded8e540d44e7bbadd8434b5d6f8f",
        3,
        "7b2b2fefc7c64d89b02a98c268b7159a",
    ),
    "PGP-stain (frozen)": (
        "965f713b7f2f452b982a46165f585dd4",
        8,
        "eeee5af20a5444f8bccec5a792301914",
    ),
    "Virchow2 (frozen)": (
        "807a761b158a4ce19fc0d984eba14028",
        4,
        "2869253ae36644679df2ec74e91b8814",
    ),
    "Virchow2-stain (frozen)": (
        "7ddd94066c78412a912a83455b2ce7f0",
        9,
        "12f1d6c1784a455b95f0df68b28c9ead",
    ),
    "GViT Stain": (
        "db6aaece63324e5a93cee722bb1f5b4e",
        10,
        "4bd9d242b2f24674a37f9e7f8e72989c",
    ),
    "GViT": ("69a386750703415f8e287e53585ee564", 5, "9d01b791da3e47f0b8d7cfd3961b4069"),
}
MIL_RUNS = {
    "Hybrid MIL PGP": (
        "4e828a9e0a59488d857e61c358e1bb70",
        12,
        "88ca0674b1c24bddab3d2e4ed65af4c2",
    ),
    "Hybrid MIL Virchow2": (
        "fc2dbe288fc04fcb8a507298e7b83375",
        11,
        "99476c7a4a114baf9bd7e7a272b904f5",
    ),
    "Classic MIL PGP": (
        "aea10f69f9a743f883861cdbf60c0a77",
        14,
        "bc5fb51707e44ad9a2efecdb8decb993",
    ),
    "Classic MIL Virchow2": (
        "5a7f3943426d4ec793653ee200adb480",
        13,
        "b1e3f5b464e646639942b64801ae4834",
    ),
}
# prefix of the logged tile-level metrics
METRIC_PREFIX = {**dict.fromkeys(TL_RUNS, "test"), **dict.fromkeys(MIL_RUNS, "tl_test")}
ALL_RUNS = {**TL_RUNS, **MIL_RUNS}
REGISTRY_URL = (
    f"{EXTERNAL_MLFLOW_URI}/#/models/Breast%20cancer%20binary%20classifier/versions"
)

# MMCI TL tables have no labels, the MIL tables do, on exactly the same tiles (identical slide/x/y sets; the joined
# labels reproduce the logged MMCI TL metrics) -> labels are taken from this MMCI MIL run
MMCI_LABELS_RUN = MIL_RUNS["Classic MIL Virchow2"][2]
TILE_KEY = ["slide", "x", "y"]

# TL test tiles with the injected DCIS coverage
# (configs/data/breast/bracs/full_rescaled_filtered_tl_test.yaml -> tiles_filtered_uri_inject_dcis_224)
DCIS_TILES_URI = (
    "mlflow-artifacts:/170/08254ee3c4ad4b36af441b11f1aaf876/artifacts/"
    "BRACS Full Rescaled & Filtered (TL Test Split)_224"
)
# DCIS tiles are excluded with the carcinoma labeling threshold
# (configs/data/breast/bracs/base.yaml -> thresholds.carcinoma_roi_t, positive if > threshold)
CARCINOMA_ROI_T = 0.5

BENIGN_COLOR, CARCINOMA_COLOR = "#2a78d6", "#eb6834"
INK, MUTED = "#0b0b0b", "#898781"

## Loading predictions and in-house thresholds

In [ ]:
@contextmanager
def internal_mlflow():
    """Temporarily switch MLflow (URI + credentials) to the internal cluster server."""
    saved_uri = mlflow.get_tracking_uri()
    saved_env = {k: os.environ.get(k) for k in MLFLOW_ENV_KEYS}

    def set_env(env: dict) -> None:
        for k, v in env.items():
            if v is None:
                os.environ.pop(k, None)
            else:
                os.environ[k] = v

    try:
        set_env(INTERNAL_MLFLOW_ENV)
        mlflow.set_tracking_uri(INTERNAL_MLFLOW_URI)
        yield
    finally:
        set_env(saved_env)
        mlflow.set_tracking_uri(saved_uri)


def read_prediction_table(path: str) -> pd.DataFrame:
    """Per-tile predictions: slide, x, y, prediction (+ target if the run has labels)."""
    return pd.read_json(path, orient="split")


def bracs_predictions(name: str) -> pd.DataFrame:
    with tempfile.TemporaryDirectory() as d:
        return read_prediction_table(
            mlflow.artifacts.download_artifacts(
                run_id=ALL_RUNS[name][0],
                artifact_path="tables/carcinoma_prediction_table.json",
                dst_path=d,
            )
        )


def _mmci_table(run_id: str) -> pd.DataFrame:
    with internal_mlflow(), tempfile.TemporaryDirectory() as d:
        # the internal server runs an older MLflow -> download by artifact URI, not by run id
        uri = mlflow.get_run(run_id).info.artifact_uri
        return read_prediction_table(
            mlflow.artifacts.download_artifacts(
                artifact_uri=f"{uri}/tables/carcinoma_prediction_table.json",
                dst_path=d,
            )
        )


_mmci_labels: pd.DataFrame | None = None


def mmci_predictions(name: str) -> pd.DataFrame:
    """MMCI test tile predictions; TL tables get the labels joined from the MMCI MIL table."""
    global _mmci_labels
    df = _mmci_table(ALL_RUNS[name][2])
    if "target" in df.columns:
        return df
    if _mmci_labels is None:
        _mmci_labels = _mmci_table(MMCI_LABELS_RUN)[[*TILE_KEY, "target"]]
    df = df.merge(_mmci_labels, on=TILE_KEY, how="left", validate="one_to_one")
    assert df["target"].notna().all(), f"{name}: MMCI tiles without a label"
    return df


def in_house_threshold(name: str) -> float:
    """Tile threshold tuned on in-house data, read from the BRACS run's resolved config."""
    with tempfile.TemporaryDirectory() as d:
        path = mlflow.artifacts.download_artifacts(
            run_id=ALL_RUNS[name][0],
            artifact_path="configs/config-resolved.yaml",
            dst_path=d,
        )
        with open(path) as f:
            return float(yaml.safe_load(f)["model_data"]["tile_threshold"])


THRESHOLDS = {name: in_house_threshold(name) for name in ALL_RUNS}
display(pd.Series(THRESHOLDS, name="in-house tile threshold").to_frame())

## BRACS vs MMCI tile score distributions

Why is the BRACS recall low despite high AUROC/AUPRC? Per model: class-wise histograms of the tile scores on BRACS
and on the in-house MMCI test set, with the in-house threshold (dashed), and a summary of where the positive scores
sit relative to it. The max-F1 threshold shows what a threshold tuned on the dataset itself would give.

In [ ]:
def best_f1_threshold(df: pd.DataFrame) -> dict:
    """Threshold maximizing F1 (positive class = carcinoma)."""
    precision, recall, thresholds = precision_recall_curve(
        df["target"].astype(bool), df["prediction"]
    )
    precision, recall = precision[:-1], recall[:-1]  # the last point has no threshold
    f1 = np.divide(
        2 * precision * recall,
        precision + recall,
        out=np.zeros_like(precision),
        where=(precision + recall) > 0,
    )
    i = int(np.argmax(f1))
    return {
        "threshold": float(thresholds[i]),
        "precision": precision[i],
        "recall": recall[i],
    }


def score_summary(df: pd.DataFrame, threshold: float) -> dict:
    y = df["target"].astype(bool)
    pos, neg = df.loc[y, "prediction"], df.loc[~y, "prediction"]
    best = best_f1_threshold(df)
    return {
        "tiles": len(df),
        "positive share": y.mean(),
        "AUROC": roc_auc_score(y, df["prediction"]),
        "AUPRC": average_precision_score(y, df["prediction"]),
        "threshold": threshold,
        "recall @ thr": (pos > threshold).mean(),
        "FPR @ thr": (neg > threshold).mean(),
        "positive median": pos.median(),
        "negative 99th pct": neg.quantile(0.99),
        "max-F1 thr": best["threshold"],
        "recall @ max-F1": best["recall"],
        "precision @ max-F1": best["precision"],
    }


def plot_class_histogram(
    ax, scores: pd.Series, threshold: float, color: str, title: str, positive: bool
) -> None:
    edges = np.linspace(0, 1, 51)
    weights = np.full(len(scores), 100 / len(scores))  # % of the tiles of this class
    ax.hist(
        scores,
        bins=edges,
        weights=weights,
        color=color,
        alpha=0.85,
        edgecolor="white",
        linewidth=0.5,
    )
    ax.axvline(threshold, color=INK, linestyle="--", linewidth=1.5)
    share = (scores > threshold).mean() if positive else (scores <= threshold).mean()
    what = "recall" if positive else "specificity"
    ax.annotate(
        f"threshold {threshold:g}\n{what} {share:.1%}",
        xy=(threshold, 0.97),
        xycoords=("data", "axes fraction"),
        xytext=(-6, 0),
        textcoords="offset points",
        ha="right",
        va="top",
        color=INK,
        fontsize=9,
    )
    ax.set_title(title, fontsize=10)
    ax.set_xlim(0, 1)
    ax.set_xlabel("predicted carcinoma probability")
    ax.set_ylabel("% of tiles in class")
    ax.grid(axis="y", color=MUTED, alpha=0.25)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)


def plot_model_histograms(
    name: str, data: dict[str, pd.DataFrame], threshold: float
) -> None:
    fig, axes = plt.subplots(2, 2, figsize=(12, 6.5), sharex=True)
    for row, (dataset, df) in zip(axes, data.items(), strict=True):
        y = df["target"].astype(bool)
        plot_class_histogram(
            row[0],
            df.loc[~y, "prediction"],
            threshold,
            BENIGN_COLOR,
            f"{dataset}: negative tiles (n={(~y).sum():,})",
            positive=False,
        )
        plot_class_histogram(
            row[1],
            df.loc[y, "prediction"],
            threshold,
            CARCINOMA_COLOR,
            f"{dataset}: positive tiles (n={y.sum():,})",
            positive=True,
        )
    fig.suptitle(f"{name}: tile score histograms (dashed = in-house threshold)", y=1.01)
    fig.tight_layout()
    plt.show()

In [ ]:
# One model at a time (the MMCI tables are large): plot, then keep only the summaries and positive-score quantiles
QUANTILES = np.linspace(0, 1, 2001)
summaries, positive_quantiles = {}, {}

for name in ALL_RUNS:
    data = {"BRACS": bracs_predictions(name), "MMCI": mmci_predictions(name)}
    for dataset, df in data.items():
        summaries[(name, dataset)] = score_summary(df, THRESHOLDS[name])
        positive_quantiles[(name, dataset)] = np.quantile(
            df.loc[df["target"].astype(bool), "prediction"], QUANTILES
        )
    plot_model_histograms(name, data, THRESHOLDS[name])
    del data

summary = pd.DataFrame(summaries).T
summary.index.names = ["model", "dataset"]

In [ ]:
# Ranking (AUROC/AUPRC) vs recall at the in-house threshold vs where the positive scores sit
display(summary.style.format({"tiles": "{:,.0f}", "threshold": "{:.2f}"}, precision=3))

In [ ]:
# Positive tiles missed at each threshold (cumulative), BRACS vs MMCI, one panel per model
ncols = 4
nrows = -(-len(ALL_RUNS) // ncols)
fig, axes = plt.subplots(
    nrows,
    ncols,
    figsize=(4 * ncols, 3.2 * nrows),
    sharex=True,
    sharey=True,
    squeeze=False,
)
# more axes than models -> not strict
for ax, name in zip(axes.flat, ALL_RUNS, strict=False):
    for dataset, color in [("BRACS", CARCINOMA_COLOR), ("MMCI", MUTED)]:
        ax.plot(
            positive_quantiles[(name, dataset)],
            QUANTILES * 100,
            color=color,
            linewidth=2,
            label=dataset,
        )
    ax.axvline(THRESHOLDS[name], color=INK, linestyle="--", linewidth=1.2)
    ax.set_xscale("logit")  # spreads out the region close to 1 where the thresholds sit
    ax.set_xlim(0.01, 0.9999)
    ax.set_ylim(0, 100)
    ax.set_title(f"{name} (thr {THRESHOLDS[name]:g})", fontsize=10)
    ax.grid(color=MUTED, alpha=0.25)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
for ax in axes.flat[len(ALL_RUNS) :]:
    ax.set_visible(False)
for ax in axes[:, 0]:
    ax.set_ylabel("% positives missed")
for ax in axes[-1, :]:
    ax.set_xlabel("threshold")
axes.flat[0].legend(frameon=False, loc="upper left")
fig.suptitle(
    "Positive tiles missed at each threshold: BRACS (orange) vs MMCI (gray); dashed = in-house threshold",
    y=1.01,
)
fig.tight_layout()
plt.show()

## Metrics without DCIS tiles

The TL test tiles were re-released with `dcis_roi_percentage` injected. Tiles whose DCIS coverage is above the
carcinoma labeling threshold (`dcis_roi_percentage` > `CARCINOMA_ROI_T`) are dropped and the tile-level metrics are
recomputed at the in-house tile thresholds (carcinoma if score > threshold).

Sanity check: on all tiles, the recomputed metrics must reproduce the metrics logged by each run
(`test/<metric>` for TL, `tl_test/<metric>` for MIL).

In [ ]:
with tempfile.TemporaryDirectory() as d:
    dcis_dir = Path(mlflow.artifacts.download_artifacts(DCIS_TILES_URI, dst_path=d))
    slides = pd.read_parquet(dcis_dir / "slides.parquet")
    tiles = pd.read_parquet(dcis_dir / "tiles.parquet")

tiles["slide"] = tiles["slide_id"].map(
    slides.set_index("id")["path"].map(lambda p: Path(p).stem)
)
is_dcis = tiles["dcis_roi_percentage"] > CARCINOMA_ROI_T
is_carcinoma = tiles["carcinoma_roi_percentage"] > CARCINOMA_ROI_T
dcis_tiles = tiles.loc[is_dcis, TILE_KEY].assign(dcis=True)
print(
    f"tiles: {len(tiles):,}, DCIS tiles: {is_dcis.sum():,} "
    f"(carcinoma positive: {(is_dcis & is_carcinoma).sum():,} of {is_carcinoma.sum():,} positive tiles)"
)
del slides, tiles, is_dcis, is_carcinoma

In [ ]:
METRICS = [
    "AUC",
    "AUPRC",
    "accuracy",
    "precision",
    "recall",
    "specificity",
    "negative_predictive_value",
]


def tl_metrics(df: pd.DataFrame, threshold: float) -> dict:
    """Tile-level metrics as logged by the test runs; carcinoma if score > threshold."""
    y = df["target"].astype(bool).to_numpy()
    pred = df["prediction"].to_numpy() > threshold
    tp, fp = int((pred & y).sum()), int((pred & ~y).sum())
    tn, fn = int((~pred & ~y).sum()), int((~pred & y).sum())

    def ratio(a: int, b: int) -> float:
        return a / b if b else 0.0

    return {
        "AUC": roc_auc_score(y, df["prediction"]),
        "AUPRC": average_precision_score(y, df["prediction"]),
        "accuracy": ratio(tp + tn, len(df)),
        "precision": ratio(tp, tp + fp),
        "recall": ratio(tp, tp + fn),
        "specificity": ratio(tn, tn + fp),
        "negative_predictive_value": ratio(tn, tn + fn),
    }


def check_logged(name: str, metrics: dict) -> None:
    """On all tiles, the recomputed metrics must match the ones logged by the run."""
    logged = mlflow.get_run(ALL_RUNS[name][0]).data.metrics
    prefix = METRIC_PREFIX[name]
    diffs = {m: abs(metrics[m] - logged[f"{prefix}/{m}"]) for m in METRICS}
    assert max(diffs.values()) < 1e-3, (
        f"{name}: recomputed metrics differ from the logged ones {diffs}"
    )


tile_counts = set()  # (all, kept, kept positive, kept negative) per model; must be the same for all models
no_dcis = {}
for name in ALL_RUNS:
    df = bracs_predictions(name).merge(
        dcis_tiles, on=TILE_KEY, how="left", validate="one_to_one"
    )
    df["dcis"] = df["dcis"].eq(True)
    assert df["dcis"].sum() == len(dcis_tiles), f"{name}: not all DCIS tiles matched"
    check_logged(name, tl_metrics(df, THRESHOLDS[name]))
    kept = df[~df["dcis"]]
    tile_counts.add(
        (
            len(df),
            len(kept),
            int(kept["target"].sum()),
            int((~kept["target"].astype(bool)).sum()),
        )
    )
    no_dcis[name] = tl_metrics(kept, THRESHOLDS[name])
    del df, kept

assert len(tile_counts) == 1, f"models were evaluated on different tiles: {tile_counts}"
N_ALL, N_KEPT, N_POS, N_NEG = tile_counts.pop()
print(
    f"tiles: {N_ALL:,} -> {N_KEPT:,} without DCIS ({N_POS:,} positive, {N_NEG:,} negative)"
)

no_dcis = pd.DataFrame(no_dcis).T[METRICS]
display(no_dcis.style.format(precision=3))

### Result pages

Rewrites the model rows of the main table (metrics logged by the runs + run links) and inserts the
"Without DCIS tiles" section before the conclusion heading of each page (an existing section is replaced). With
`PAGES_DIR = None` the tables are only displayed.

In [ ]:
SECTION = "## Without DCIS tiles"
CONCLUSION = "## Conclusion(s)"
INTRO = (
    f"Same models, tiles and per-model tile thresholds as above, but tiles with DCIS coverage above the carcinoma "
    f"threshold (`dcis_roi_percentage` > {CARCINOMA_ROI_T}) are excluded ({N_ALL - N_KEPT:,} tiles; all of them are "
    f"carcinoma positive). Remaining: {N_KEPT:,} tiles ({N_POS:,} positive, {N_NEG:,} negative)."
)


def row(name: str, values: dict) -> str:
    run_id, version, _ = ALL_RUNS[name]
    return (
        f"| [{name}]({REGISTRY_URL}/{version}) | "
        + " | ".join(f"{values[m]:.3f}" for m in METRICS)
        + f" | [URL]({EXTERNAL_MLFLOW_URI}/#/experiments/170/runs/{run_id}) |"
    )


def logged_metrics(name: str) -> dict:
    logged = mlflow.get_run(ALL_RUNS[name][0]).data.metrics
    return {m: logged[f"{METRIC_PREFIX[name]}/{m}"] for m in METRICS}


def update_page(page: str, models: list[str], auprc_label: str) -> None:
    header = (
        f"| Model | AUROC | {auprc_label} | Accuracy | Precision | Recall | Specificity | Negative Predictive Score "
        "| MLFlow Run |"
    )
    main_rows = {name: row(name, logged_metrics(name)) for name in models}
    section = [
        SECTION,
        "",
        INTRO,
        "",
        header,
        "| --- | --- | --- | --- | --- | --- | --- | --- | --- |",
        *(row(name, no_dcis.loc[name]) for name in models),
    ]
    display(
        Markdown(
            "\n".join(
                [
                    "### Main table",
                    "",
                    header,
                    "| --- " * 9 + "|",
                    *main_rows.values(),
                    "",
                    *section,
                ]
            )
        )
    )
    if PAGES_DIR is None:
        return

    path = Path(PAGES_DIR) / page
    lines = path.read_text().splitlines()
    main_end = next(
        i for i, line in enumerate(lines) if line.startswith("## ")
    )  # main table: before the 1st section
    for name, new_row in main_rows.items():
        # whitespace-insensitive match of the row name
        i = next(
            i
            for i, line in enumerate(lines[:main_end])
            if " ".join(line.split()).startswith(f"| [{name}](")
        )
        lines[i] = new_row
    text = "\n".join(lines) + "\n"
    if SECTION in text:  # drop the old section, up to the next "## " heading
        start = text.index(SECTION)
        end = text.find("\n## ", start + len(SECTION))
        text = text[:start] + (text[end + 1 :] if end != -1 else "")
    assert CONCLUSION in text, f"{page}: heading {CONCLUSION!r} not found"
    path.write_text(
        text.replace(CONCLUSION, "\n".join(section) + "\n\n" + CONCLUSION, 1)
    )
    print(f"updated {page}")


update_page("bracs_tl_eval.md", list(TL_RUNS), "AUPRC")
update_page("bracs_mil_eval.md", list(MIL_RUNS), "AUPR")